# 🔬 Rendering Surface from SPH Particle Snapshots

This notebook documents the full pipeline for extracting surfaces, as examplifed by the oxygen ionisation fronts from an SPH simulation and visualising them as animated meshes in Blender.

---

## 📦 Pipeline Overview
```
Simulation Snapshot (.hdf5) --> yt  (dataset loading) --> AstroVis (Data Processing) --> Surface Data (.npz file) --> Blender


## 1. Imports & Data Loading

In [ ]:
from AstroVis.backend import * ## For Surface Calculation
import yt

: 

## 2. Load Simulation Dataset and SPH Particle

Load the HDF5 snapshot with `yt`, then generate ionisation fraction fields for oxygen species.  
`generate_species_fraction_fields(ds, "O")` scans the dataset and returns getter functions for all available oxygen ions (OI, OII, OIII, OIV, …).
`load_particles()` reads gas particles from `PartType0` along with the oxygen fraction fields attached above.  
The result `stromgren` holds both particle positions and all requested field arrays.

In [ ]:
## Loading the data from dataset

ds = yt.load("output_7bin_HHeZdust_real_0060.hdf5")

fields = generate_species_fraction_fields(ds, "O")  # Generate getter functions for O species fractions
stromgren = load_particles(ds, ptype="PartType0", fields=fields)
print(stromgren)

## 3. Data Processing

For each oxygen species fraction, we

1. Convert SPH particle Data into 3D Grid by sph_to_grid
2. Extract the ionization surface by grid_to_surface() or grid_to_ridge_surface()
3. Save as .npz file

Note: Two different pipelines are used depending on the ion:

| Species | Resolution | Surface Method | Reason |
|---|---|---|---|
| `OI`, `OIV` | 256³ | `grid_to_surface` (isosurface at threshold 0.5) | Sharp, well-defined ionisation boundaries |
| `OII`, `OIII` | 128³ | `grid_to_ridge_surface` | Diffuse transition zones — ridge extraction captures the gradient peak |

In [ ]:
grid = {}
surface = {}

for species, fracion in stromgren.fields.items():
    grid[species] = sph_to_grid(stromgren, species)
    
    if species == "OI_fraction" or species == "OIV_fraction":
        surface[species] = grid_to_surface(grid[species], threshold=0.5)
        save(f"{species}_surface.npz", surface[species])
        
    elif species == "OII_fraction" or species == "OIII_fraction":
        surface[f"{species}_inner"] = grid_to_surface_local_max(grid[species],outer=False)
        save(f"{species}_inner.npz", surface[f"{species}_inner"])
        
        surface[f"{species}_outer"] = grid_to_surface_local_max(grid[species],outer=True)
        save(f"{species}_outer.npz", surface[f"{species}_outer"])

    else:
        continue


    
    

## 4. Setup Animation in Blender 

> ⚠️ Run the cells below inside Blender's **Python Console** or **Text Editor** — `bpy` is only available there.

Each `.npz` file (saved in Section 3) is loaded back and registered as an animated mesh in Blender.

| Variable | File | Shader | `position_scale` |
|---|---|---|---|
| `OI` | `OI_fraction_surface.npz` | `OII_trans` | 256 | 
| `OII` | `OII_fraction_surface.npz` | `OII_trans` | 256 | 
| `OIII` | `OIII_fraction_surface.npz` | `OIII_trans` | 256 | 
| `OIV` | `OIV_fraction_surface.npz` | `OIV_trans` | 256 | 

`create_mesh_shader()` create a translucent material for ions.
`setup_mesh_animation()` creates a Blender mesh object and keyframes vertex positions across frames.  
`set_object_shader()` assigns the shader to the object.

In [ ]:
surface_folder = r"Path\To\Surface\Files" ## Change this to the actual path where the surface files are stored

create_mesh_shader(['OI_surface', 'OII_surface', 'OIII_surface', 'OIV_surface'])

OI_data = load(rf"{surface_folder}\OI_fraction_surface.npz")
OI      = setup_mesh_animation(OI_data,  "OI_surface",   position_scale=256)
set_object_shader(OI,   "OI_surface_meshshader")

OII_data = load(rf"{surface_folder}\OII_fraction_surface.npz")
OII      = setup_mesh_animation(OII_data, "OII_surface",   position_scale=256)
set_object_shader(OII,  "OII_surface_meshshader")

OIII_data = load(rf"{surface_folder}\OIII_fraction_surface.npz")
OIII      = setup_mesh_animation(OIII_data, "OIII_surface", position_scale=256)
set_object_shader(OIII, "OIII_surface_meshshader")

OIV_data = load(rf"{surface_folder}\OIV_fraction_surface.npz")
OIV      = setup_mesh_animation(OIV_data,  "OIV_surface", position_scale=256)
set_object_shader(OIV,  "OIV_surface_meshshader")